In [ ]:
import os
import shutil

# 1. Setup paths
DATASET_NAME = 'ashery/chexpert'
DOWNLOAD_DIR = '/content/chexpert_dataset'

# 2. Clean start: Remove old corrupted data
if os.path.exists(DOWNLOAD_DIR):
    print("Cleaning up old dataset directory...")
    shutil.rmtree(DOWNLOAD_DIR)
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# 3. Download the dataset
print("Downloading dataset (11GB)... This may take 2-3 minutes.")
!kaggle datasets download -d {DATASET_NAME} --path {DOWNLOAD_DIR}

# 4. Extract using a more memory-efficient approach
zip_path = os.path.join(DOWNLOAD_DIR, 'chexpert.zip')
print("Extracting... Please wait.")
# Using -n to not overwrite if exists, and -q for quiet (less memory pressure on logs)
!unzip -q {zip_path} -d {DOWNLOAD_DIR}

# 5. Final Verification
image_count = 0
for root, dirs, files in os.walk(DOWNLOAD_DIR):
    image_count += len([f for f in files if f.endswith('.jpg')])

print(f"\nSuccess! Total images found: {image_count}")
if image_count >= 223000:
    print("Dataset is complete. You can now proceed to load CSVs.")
    os.remove(zip_path) # Safe to remove zip now
else:
    print("Warning: Image count is still low. Check for disk space errors.")

Cleaning up old dataset directory...
Dataset URL: https://www.kaggle.com/datasets/ashery/chexpert
License(s): CC0-1.0
100% 10.7G/10.7G [04:39<00:00, 41.1MB/s]

Extracting... Please wait.

Success! Total images found: 223649
Dataset is complete. You can now proceed to load CSVs.


In [40]:
import tensorflow as tf
# Enable Mixed Precision for faster GPU training
from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print('Mixed precision policy set to:', policy.name)

Mixed precision policy set to: mixed_float16


In [41]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AutoTune = tf.data.AUTOTUNE

In [42]:
def load_preprocess_img(img_path,label):
  img_bytes = tf.io.read_file(img_path)
  img = tf.image.decode_jpeg(img_bytes,channels=3)
  img = tf.image.resize(img,IMG_SIZE,method='bilinear')
  img= tf.cast(img,tf.float32)/255.0
  return img, label

In [43]:
def create_fast_dataset(paths, labels, is_training=False):
  ds = tf.data.Dataset.from_tensor_slices((paths, labels))
  if is_training:
    ds = ds.shuffle(buffer_size=10000)
  ds = ds.map(load_preprocess_img, num_parallel_calls=AutoTune)
  ds = ds.batch(BATCH_SIZE)
  ds = ds.prefetch(buffer_size=AutoTune)
  return ds

First, let's load the `train.csv` and `valid.csv` files from the downloaded dataset. We will assume the paths in the CSV are relative to `chexpert_dataset/` and need to be joined with the base path.

In [44]:
import pandas as pd
import numpy as np
import os

# The CSV files (train.csv, valid.csv) are located directly inside DOWNLOAD_DIR
# Load the training and validation CSV files
train_df = pd.read_csv(os.path.join(DOWNLOAD_DIR, 'train.csv'))
valid_df = pd.read_csv(os.path.join(DOWNLOAD_DIR, 'valid.csv'))

# Define the base path for the images within the dataset directory.
# The 'Path' column in the CSVs has paths like 'CheXpert-v1.0-small/train/...' which are relative to DOWNLOAD_DIR.
# This variable was not used for CSV loading but might be useful later.
base_data_path = os.path.join(DOWNLOAD_DIR, 'CheXpert-v1.0-small')

# Display the first few rows of the dataframes to understand their structure
print("Train DataFrame head:")
display(train_df.head())
print("\nValidation DataFrame head:")
display(valid_df.head())

Train DataFrame head:


,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
0,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68,Frontal,AP,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,1.0
1,CheXpert-v1.0-small/train/patient00002/study2/...,Female,87,Frontal,AP,NaN,NaN,-1.0,1.0,NaN,-1.0,-1.0,NaN,-1.0,NaN,-1.0,NaN,1.0,NaN
2,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,Frontal,AP,NaN,NaN,NaN,1.0,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN
3,CheXpert-v1.0-small/train/patient00002/study1/...,Female,83,Lateral,NaN,NaN,NaN,NaN,1.0,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,1.0,NaN
4,CheXpert-v1.0-small/train/patient00003/study1/...,Male,41,Frontal,AP,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN



Validation DataFrame head:


,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
0,CheXpert-v1.0-small/valid/patient64541/study1/...,Male,73,Frontal,AP,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,CheXpert-v1.0-small/valid/patient64542/study1/...,Male,70,Frontal,PA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,CheXpert-v1.0-small/valid/patient64542/study1/...,Male,70,Lateral,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,CheXpert-v1.0-small/valid/patient64543/study1/...,Male,85,Frontal,AP,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,CheXpert-v1.0-small/valid/patient64544/study1/...,Female,42,Frontal,AP,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


The `Path` column contains the relative paths to the images. We need to prepend the full base path to these. For labels, let's select a few common pathologies to demonstrate, handling the -1 (uncertain) and NaN (not mentioned) values. For simplicity, we'll convert -1 to 0 (negative) and fill NaN with 0 for now. You can adjust this strategy based on your specific modeling needs.

In [45]:
# Update image paths to be absolute paths starting from /content/
# The CSVs contain paths like 'CheXpert-v1.0-small/train/patient00001/...'
# Since we extracted everything into /content/chexpert_dataset, we map them correctly.

def fix_path(path):
    # Remove the prefix 'CheXpert-v1.0-small/' if it exists and join with absolute DOWNLOAD_DIR
    relative_path = path.replace('CheXpert-v1.0-small/', '')
    return os.path.join(DOWNLOAD_DIR, relative_path)

train_df['Path'] = train_df['Path'].apply(fix_path)
valid_df['Path'] = valid_df['Path'].apply(fix_path)

# Target labels (14 pathologies)
TARGET_LABELS = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
    'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
    'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices'
]

# Clean labels: Fill NaN with 0 and treat uncertain (-1) as negative (0)
for label in TARGET_LABELS:
    train_df[label] = train_df[label].fillna(0).replace(-1, 0)
    valid_df[label] = valid_df[label].fillna(0).replace(-1, 0)

train_paths = train_df['Path'].values
train_labels = train_df[TARGET_LABELS].values
valid_paths = valid_df['Path'].values
valid_labels = valid_df[TARGET_LABELS].values

print(f"Verified Path Example: {train_paths[0]}")
print(f"File exists? {os.path.exists(train_paths[0])}")

Verified Path Example: /content/chexpert_dataset/train/patient00001/study1/view1_frontal.jpg
File exists? True


Now, we can use the `create_fast_dataset` function to build our TensorFlow datasets for training and validation.

In [46]:
# Create TensorFlow datasets using the verified absolute paths
train_ds = create_fast_dataset(train_paths, train_labels, is_training=True)
valid_ds = create_fast_dataset(valid_paths, valid_labels, is_training=False)

try:
    # Verify a batch from the training dataset
    print("Attempting to read the first batch...")
    for images, labels in train_ds.take(1):
        print(f"Image batch shape: {images.shape}")
        print(f"Label batch shape: {labels.shape}")
        break
    print("Successfully created and verified datasets!")
except tf.errors.NotFoundError as e:
    print(f"\nERROR: Files still not found at the expected paths.\nDetails: {e}")

Attempting to read the first batch...
Image batch shape: (32, 224, 224, 3)
Label batch shape: (32, 14)
Successfully created and verified datasets!


In [47]:
from tensorflow.keras import layers, models, applications

In [48]:
@tf.keras.utils.register_keras_serializable(package='CustomLayers')
class GraphAttentionLayer(layers.Layer):
    def __init__(self, embed_dim, num_heads=4, **kwargs):
        super(GraphAttentionLayer, self).__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.multi_head_att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.layernorm = layers.LayerNormalization()
        self.add = layers.Add()

    def call(self, x):
        att_output = self.multi_head_att(query=x, value=x, key=x)
        x = self.add([x, att_output])
        return self.layernorm(x)

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
        })
        return config

In [49]:
def hybrid_chexpert_model(input_shape=(224, 224, 3), num=14, embed_dim=128):
    inputs = layers.Input(input_shape, name="chest_xray")

    # 1. Advanced CNN Backbone (EfficientNetB3)
    base_model = applications.EfficientNetB3(
        include_top=False, weights='imagenet', input_shape=input_shape, pooling='avg'
    )
    base_model.trainable = False
    global_features = base_model(inputs)

    # ADDED: Batch Normalization for faster convergence and stability
    global_features = layers.BatchNormalization()(global_features)

    # 2. Disease Node Initialization
    x = layers.Dense(num * embed_dim)(global_features)
    x = layers.Activation('swish')(x)
    x = layers.LayerNormalization()(x) # Normalize before reshaping to nodes
    disease_nodes = layers.Reshape((num, embed_dim))(x)

    # 3. Dynamic Graph Attention Processing
    x = GraphAttentionLayer(embed_dim)(disease_nodes)
    x = GraphAttentionLayer(embed_dim)(x)

    # 4. Feature Refinement
    se = layers.GlobalAveragePooling1D()(x)
    se = layers.Dense(embed_dim // 8, activation='relu')(se)
    se = layers.Dense(embed_dim, activation='sigmoid')(se)
    se = layers.Reshape((1, embed_dim))(se)
    x = layers.Multiply()([x, se])

    # 5. Final Classification Head
    x = layers.Dense(64, activation='swish')(x)
    x = layers.BatchNormalization()(x) # Final normalization before logits
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(1)(x)
    outputs = layers.Flatten()(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="Improved_GAT_CheXpert_Optimized")
    return model

In [50]:
# Re-instantiating the upgraded model
model = hybrid_chexpert_model()
model.summary()

Model: "Improved_GAT_CheXpert_Optimized"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ chest_xray          │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficientnetb3      │ (None, 1536)      │ 10,783,535 │ chest_xray[0][0]  │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1536)      │      6,144 │ efficientnetb3[0… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 1792)      │  2,754,304 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 1792)      │          0 │ dense_15[0][0]    │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1792)      │      3,584 │ activation_2[0][… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_6 (Reshape) │ (None, 14, 128)   │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ graph_attention_la… │ (None, 14, 128)   │    264,064 │ reshape_6[0][0]   │
│ (GraphAttentionLay… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ graph_attention_la… │ (None, 14, 128)   │    264,064 │ graph_attention_… │
│ (GraphAttentionLay… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ graph_attention_… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 16)        │      2,064 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_17 (Dense)    │ (None, 128)       │      2,176 │ dense_16[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_7 (Reshape) │ (None, 1, 128)    │          0 │ dense_17[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_3          │ (None, 14, 128)   │          0 │ graph_attention_… │
│ (Multiply)          │                   │            │ reshape_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_18 (Dense)    │ (None, 14, 64)    │      8,256 │ multiply_3[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 14, 64)    │        256 │ dense_18[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_17          │ (None, 14, 64)    │          0 │ batch_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 14, 1)     │         65 │ dropout_17[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 14)        │          0 │ dense_19[0][0]    │
└─────────────────────┴───────────────────┴────────────┴─────────────────

 Total params: 14,088,512 (53.74 MB)

 Trainable params: 3,301,777 (12.60 MB)

 Non-trainable params: 10,786,735 (41.15 MB)

In [51]:
# 1. Calculate weights
pos_counts = np.sum(train_labels, axis=0)
total_samples = len(train_labels)
neg_counts = total_samples - pos_counts
pos_weights = np.sqrt(neg_counts / (pos_counts + 1e-7))
pos_weights_tensor = tf.constant(pos_weights, dtype=tf.float32)

# 2. Define and Register Loss Function (without @tf.function wrapper to prevent loading errors)
@tf.keras.utils.register_keras_serializable(package='CustomLoss')
def weighted_bce_loss(y_true, y_pred_logits):
    y_true = tf.cast(y_true, tf.float32)
    loss = tf.nn.weighted_cross_entropy_with_logits(
        labels=y_true,
        logits=y_pred_logits,
        pos_weight=pos_weights_tensor
    )
    return tf.reduce_mean(loss)

In [52]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=weighted_bce_loss,
    # jit_compile=True can further speed up training on compatible GPUs
    jit_compile=True,
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='accuracy', threshold=0.0),
        tf.keras.metrics.AUC(name='roc_auc', multi_label=True, num_labels=14, from_logits=True),
        tf.keras.metrics.AUC(name='pr_auc', curve='PR', multi_label=True, num_labels=14, from_logits=True),
        tf.keras.metrics.Precision(name='precision', thresholds=0.0),
        tf.keras.metrics.Recall(name='recall', thresholds=0.0)
    ]
)

In [55]:
# 1. Instantiate the model
model = hybrid_chexpert_model()

# 2. Compile with Maximum GPU Optimizations
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=weighted_bce_loss,
    jit_compile=True, # Enable XLA for massive speedup on GPU
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='accuracy', threshold=0.0),
        tf.keras.metrics.AUC(name='roc_auc', multi_label=True, num_labels=14, from_logits=True),
        tf.keras.metrics.AUC(name='pr_auc', curve='PR', multi_label=True, num_labels=14, from_logits=True),
        # Added more evaluation metrics for real-time monitoring
        tf.keras.metrics.Precision(name='precision', thresholds=0.0),
        tf.keras.metrics.Recall(name='recall', thresholds=0.0)
    ]
)

# 3. Optimized Callbacks
callbacks_list = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_pr_auc', patience=7, restore_best_weights=True, mode='max'
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath='gat_chexpert_epoch_{epoch:02d}.keras', save_best_only=False
    )
]

# 4. Start Optimized Training
EPOCHS = 20
history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS,
    callbacks=callbacks_list
)

Epoch 1/20
 124/6982 ━━━━━━━━━━━━━━━━━━━━ 10:19 90ms/step - accuracy: 0.6357 - loss: 0.8899 - pr_auc: 0.1559 - precision: 0.2202 - recall: 0.5191 - roc_auc: 0.4935

KeyboardInterrupt: 

thresh_tune

In [38]:
from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np

# 1. Extract true validation labels
y_val_true = np.concatenate([y for x, y in valid_ds], axis=0)

# 2. Get model predictions and convert logits to probabilities with Sigmoid
val_logits = model.predict(valid_ds)
val_probs = tf.nn.sigmoid(val_logits).numpy()

# 3. Search for optimal threshold for each disease
optimal_thresholds = []
candidate_thresholds = np.linspace(0.1, 0.9, 81) # Test thresholds from 0.1 to 0.9

print("Disease Index | Best Threshold | Best F1 | Precision | Recall")
print("-" * 65)

for i in range(14):
    best_thresh = 0.5
    best_f1 = 0.0

    for thresh in candidate_thresholds:
        preds = (val_probs[:, i] >= thresh).astype(int)
        score = f1_score(y_val_true[:, i], preds, zero_division=0)

        if score > best_f1:
            best_f1 = score
            best_thresh = thresh

    optimal_thresholds.append(best_thresh)

    # Evaluate on the best threshold
    final_preds = (val_probs[:, i] >= best_thresh).astype(int)
    p = precision_score(y_val_true[:, i], final_preds, zero_division=0)
    r = recall_score(y_val_true[:, i], final_preds, zero_division=0)

    print(f"Disease {i:02d}    | {best_thresh:.3f}          | {best_f1:.3f}   | {p:.3f}     | {r:.3f}")

optimal_thresholds = np.array(optimal_thresholds)

8/8 ━━━━━━━━━━━━━━━━━━━━ 69s 5s/step
Disease Index | Best Threshold | Best F1 | Precision | Recall
-----------------------------------------------------------------
Disease 00    | 0.100          | 0.279   | 0.162     | 1.000
Disease 01    | 0.100          | 0.636   | 0.466     | 1.000
Disease 02    | 0.200          | 0.452   | 0.292     | 1.000
Disease 03    | 0.100          | 0.700   | 0.538     | 1.000
Disease 04    | 0.110          | 0.028   | 0.014     | 1.000
Disease 05    | 0.330          | 0.324   | 0.193     | 1.000
Disease 06    | 0.200          | 0.284   | 0.180     | 0.667
Disease 07    | 0.100          | 0.066   | 0.034     | 1.000
Disease 08    | 0.320          | 0.526   | 0.362     | 0.963
Disease 09    | 0.100          | 0.066   | 0.034     | 1.000
Disease 10    | 0.380          | 0.463   | 0.314     | 0.881
Disease 11    | 0.110          | 0.045   | 0.023     | 1.000
Disease 12    | 0.500          | 0.000   | 0.000     | 0.000
Disease 13    | 0.100          | 0.628   |

In [39]:
# ==========================================
# Final Evaluation on Validation Set (acting as Test Set)
# Note: Since a separate `test_ds` is not defined, `valid_ds` is used for final evaluation.
# This may lead to optimistically biased metrics as thresholds were tuned on the same set.
# ==========================================

# 1. Get model logits for the validation data
test_logits = model.predict(valid_ds)

# 2. Convert logits to probabilities (between 0 and 1)
test_probs = tf.nn.sigmoid(test_logits).numpy()

# 3. Apply disease-specific optimal thresholds
# The output will be a 0/1 (False/True) matrix with dimensions (num_samples, 14)
final_predictions = (test_probs >= optimal_thresholds).astype(int)

# Now, calculate final metrics like F1, Precision, and Recall on the validation data:
from sklearn.metrics import classification_report
y_test_true = np.concatenate([y for x, y in valid_ds], axis=0)

print(classification_report(y_test_true, final_predictions, zero_division=0))

8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 84ms/step
              precision    recall  f1-score   support

           0       0.16      1.00      0.28        38
           1       0.47      1.00      0.64       109
           2       0.29      1.00      0.45        68
           3       0.54      1.00      0.70       126
           4       0.01      1.00      0.03         1
           5       0.19      1.00      0.32        45
           6       0.18      0.67      0.28        33
           7       0.03      1.00      0.07         8
           8       0.36      0.96      0.53        80
           9       0.03      1.00      0.07         8
          10       0.31      0.88      0.46        67
          11       0.02      1.00      0.05         1
          12       0.00      0.00      0.00         0
          13       0.46      1.00      0.63       107

   micro avg       0.27      0.97      0.42       691
   macro avg       0.22      0.89      0.32       691
weighted avg       0.37      0.97      0.5

### Function to Get Binary Predictions Using Optimal Thresholds

To consistently apply the `optimal_thresholds` found during validation, we can create a helper function that performs the prediction, converts logits to probabilities, and then applies the disease-specific thresholds. This ensures that any time you want to get final binary predictions from your model, you use the established optimal thresholds.

In [54]:
def get_binary_predictions(model, dataset, optimal_thresholds):
    """
    Generates binary predictions from a model using pre-calculated optimal thresholds.

    Args:
        model: The trained Keras model.
        dataset: A tf.data.Dataset (e.g., valid_ds, test_ds) containing images.
        optimal_thresholds: A numpy array of shape (num_classes,) containing
                            the optimal threshold for each class.

    Returns:
        A numpy array of binary predictions (0s and 1s) with shape (num_samples, num_classes).
    """
    # 1. Get model logits for the given dataset
    logits = model.predict(dataset)

    # 2. Convert logits to probabilities (between 0 and 1) using sigmoid activation
    probabilities = tf.nn.sigmoid(logits).numpy()

    # 3. Apply the disease-specific optimal thresholds
    # This compares each probability to its corresponding optimal threshold
    # and converts it to a binary (0 or 1) prediction.
    binary_predictions = (probabilities >= optimal_thresholds).astype(int)

    return binary_predictions

# Example usage with your validation dataset:
print("Generating binary predictions for the validation set using optimal thresholds...")
final_val_predictions_optimal = get_binary_predictions(model, valid_ds, optimal_thresholds)

print("First 5 binary predictions (for first sample):")
print(final_val_predictions_optimal[:5, :])

Generating binary predictions for the validation set using optimal thresholds...
8/8 ━━━━━━━━━━━━━━━━━━━━ 70s 5s/step
First 5 binary predictions (for first sample):
[[1 1 0 1 1 0 1 1 0 1 0 1 0 1]
 [1 1 0 1 1 0 1 1 0 1 0 1 0 1]
 [1 1 1 1 1 0 0 1 0 1 1 1 0 1]
 [1 1 0 1 1 1 1 1 0 1 1 1 0 1]
 [1 1 0 1 1 0 0 1 0 1 1 1 0 1]]


### Loading and Evaluating Model Checkpoint `gat_chexpert_epoch_14.keras`

To evaluate a specific checkpoint, we first need to load it. Since your model uses custom layers and a custom loss function, we'll need to provide these to `tf.keras.models.load_model`.

In [60]:
# Load the model from the specified checkpoint
# Ensure custom objects (GraphAttentionLayer, weighted_bce_loss) are available in the current scope.
# These should have been defined and executed in earlier cells (o_zQS-PSn2Zt and X-jk-qebvUZV).

print("Loading model from gat_chexpert_epoch_14.keras...")
loaded_model = tf.keras.models.load_model(
    'gat_chexpert_epoch_14.keras',
    custom_objects={
        'GraphAttentionLayer': GraphAttentionLayer,
        'weighted_bce_loss': weighted_bce_loss
    },
    compile=False # Load model architecture and weights, but skip compiling with saved optimizer/loss/metrics
)
print("Model 'gat_chexpert_epoch_14.keras' loaded successfully!")

Loading model from gat_chexpert_epoch_14.keras...


/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'graph_attention_layer_20', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'graph_attention_layer_21', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model 'gat_chexpert_epoch_14.keras' loaded successfully!


In [62]:
# Recompile the loaded model with the correct custom loss and metrics
# The original compilation parameters are in cell I3h6pFQbv2cx.
print("Recompiling the loaded model...")
loaded_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), # Assuming a learning rate, adjust if needed
    loss=weighted_bce_loss,
    jit_compile=True,
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='accuracy', threshold=0.0),
        tf.keras.metrics.AUC(name='roc_auc', multi_label=True, num_labels=14, from_logits=True),
        tf.keras.metrics.AUC(name='pr_auc', curve='PR', multi_label=True, num_labels=14, from_logits=True),
        tf.keras.metrics.Precision(name='precision', thresholds=0.0),
        tf.keras.metrics.Recall(name='recall', thresholds=0.0)
    ]
)
print("Loaded model recompiled successfully!")

Recompiling the loaded model...
Loaded model recompiled successfully!


Now that the model is loaded, we can proceed with evaluating its performance on the `valid_ds` using the `optimal_thresholds`.

In [63]:
from sklearn.metrics import classification_report, roc_auc_score

# Get binary predictions using the loaded model and optimal thresholds
print("Generating binary predictions for the validation set using the loaded model...")
final_predictions_loaded_model = get_binary_predictions(loaded_model, valid_ds, optimal_thresholds)

# Get true labels for the validation set
y_val_true = np.concatenate([y for x, y in valid_ds], axis=0)

print("\n--- Classification Report for gat_chexpert_epoch_14.keras ---")
print(classification_report(y_val_true, final_predictions_loaded_model, zero_division=0))

# Additionally, calculate AUC scores for the loaded model
print("\n--- AUC Scores for gat_chexpert_epoch_14.keras ---")
# For AUC, we need probabilities, not binary predictions
val_logits_loaded_model = loaded_model.predict(valid_ds)
val_probs_loaded_model = tf.nn.sigmoid(val_logits_loaded_model).numpy()

# Calculate macro-averaged ROC AUC
roc_auc_macro = roc_auc_score(y_val_true, val_probs_loaded_model, average='macro')
print(f"Macro-averaged ROC AUC: {roc_auc_macro:.4f}")

# Calculate per-class ROC AUC if needed
# roc_auc_per_class = [roc_auc_score(y_val_true[:, i], val_probs_loaded_model[:, i]) for i in range(y_val_true.shape[1])]
# for i, auc_score in enumerate(roc_auc_per_class):
#     print(f"ROC AUC for class {i:02d}: {auc_score:.4f}")

Generating binary predictions for the validation set using the loaded model...
8/8 ━━━━━━━━━━━━━━━━━━━━ 74s 5s/step

--- Classification Report for gat_chexpert_epoch_14.keras ---
              precision    recall  f1-score   support

           0       0.16      1.00      0.28        38
           1       0.47      1.00      0.64       109
           2       0.29      1.00      0.45        68
           3       0.54      1.00      0.70       126
           4       0.00      1.00      0.01         1
           5       0.19      1.00      0.32        45
           6       0.14      1.00      0.25        33
           7       0.03      1.00      0.07         8
           8       0.39      0.14      0.20        80
           9       0.03      1.00      0.07         8
          10       0.29      1.00      0.45        67
          11       0.00      0.00      0.00         1
          12       0.00      0.00      0.00         0
          13       0.46      1.00      0.63       107

   micro 

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


You can now use this `get_binary_predictions` function whenever you need to get the final, thresholded predictions from your `model`.

### Loading and Evaluating Model Checkpoint `gat_chexpert_epoch_14.keras`

To evaluate a specific checkpoint, we first need to load it. Since your model uses custom layers and a custom loss function, we'll need to provide these to `tf.keras.models.load_model`.

Now that the model is loaded, we can proceed with evaluating its performance on the `valid_ds` using the `optimal_thresholds`.

In [57]:
from sklearn.metrics import classification_report, roc_auc_score

# Get binary predictions using the loaded model and optimal thresholds
print("Generating binary predictions for the validation set using the loaded model...")
final_predictions_loaded_model = get_binary_predictions(loaded_model, valid_ds, optimal_thresholds)

# Get true labels for the validation set
y_val_true = np.concatenate([y for x, y in valid_ds], axis=0)

print("\n--- Classification Report for gat_chexpert_epoch_14.keras ---")
print(classification_report(y_val_true, final_predictions_loaded_model, zero_division=0))

# Additionally, calculate AUC scores for the loaded model
print("\n--- AUC Scores for gat_chexpert_epoch_14.keras ---")
# For AUC, we need probabilities, not binary predictions
val_logits_loaded_model = loaded_model.predict(valid_ds)
val_probs_loaded_model = tf.nn.sigmoid(val_logits_loaded_model).numpy()

# Calculate macro-averaged ROC AUC
roc_auc_macro = roc_auc_score(y_val_true, val_probs_loaded_model, average='macro')
print(f"Macro-averaged ROC AUC: {roc_auc_macro:.4f}")

# Calculate per-class ROC AUC if needed
# roc_auc_per_class = [roc_auc_score(y_val_true[:, i], val_probs_loaded_model[:, i]) for i in range(y_val_true.shape[1])]
# for i, auc_score in enumerate(roc_auc_per_class):
#     print(f"ROC AUC for class {i:02d}: {auc_score:.4f}")

Generating binary predictions for the validation set using the loaded model...


NameError: name 'loaded_model' is not defined

You can now use this `get_binary_predictions` function whenever you need to get the final, thresholded predictions from your `model`.